# TD6 — Exercice 1 : Variogramme 1D du prix de l'or

**Traitement statistique, géostatistique et approche systèmes — ENSG 2A (S7)**

*Auteurs: Guillaume Caumon, Oumayma Razik*

On se propose d'étudier les statistiques et la variabilité temporelle du cours de l'or (variable **`GoldPrice`**), en dollars par once ($/once)** du 01/01/1982 au 09/10/1997 (4 174 valeurs). L'objectif est de cartactériser la volatilité, c'est à dire la vitesse et la fréquence auxquelle le cours de cette "valeur refuge" tend à évoluer sur la période.

Nous allons commencer par étudier la série temporellle et sa distribution avant de calculer le variogramme expérimental.

## 1. Chargement et visualisation des données

On charge le fichier `prix_or_1D.csv` contenant les 4 174 valeurs du prix de l'or.
Avant d'étudier la distribution statistique, on commence par charger et vérifier les données.

In [ ]:
import sys
import os
from ipaddress import ip_network

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import gstlearn as gl
import gstlearn.plot as gp
import ipywidgets as widgets
from ipywidgets import interactive_output
from pandas.core.interchange.dataframe_protocol import DataFrame

from geostat_utils import ResultsDir

# Ajout du chemin d'importation pour le module geostat_utils 
sys.path.append(os.path.abspath(os.path.join("..")))
import geostat_utils as gsutils

# Set output directory for figures
results = gsutils.ResultsDir("/TD6_outputs")

# Load the data and set the data as index to make date-based queries easy
gold_price: pd.DataFrame = pd.read_csv('../data/prix_or_1D.csv', header=0)
gold_price["Date"] = pd.to_datetime(gold_price["Date"], dayfirst=True)
gold_price.set_index("Date", inplace=True)
print(gold_price)

Regardons à présent l'histogramme:

In [ ]:
def get_period_as_string(df: pd.DataFrame, start_date: str = None, end_date: str = None) -> str:
    actual_start = start_date if start_date else df.index.min().strftime('%Y-%m-%d')
    actual_end = end_date if end_date else df.index.max().strftime('%Y-%m-%d')
    return f"de {actual_start} à {actual_end}"

def plot_histogram(df: pd.DataFrame, field: str,
                   unit = "", start_date: str = None, end_date: str = None):
    """
    Plots a histogram and its corresponding CDF for a DataFrame column.
    Statistics (count, min, percentiles, mean, std) are displayed on the histogram subplot.

    :param df: Input DataFrame containing the data to plot.
    :param field: Name of the column to analyze and plot.
    :param unit: Unit of the data (displayed in the title and x-axis label). Defaults to "".
    :param start_date: Start date for the period label (format 'YYYY-MM-DD').
                       If None, uses the minimum date in the DataFrame index. Defaults to None.
    :param end_date: End date for the period label (format 'YYYY-MM-DD').
                     If None, uses the maximum date in the DataFrame index. Defaults to None.
    :Notes:
       - NaN values in the specified column are automatically dropped.
       - The histogram uses 30 bins by default (linspaced between min and max values).
       - The CDF is normalized to [0, 1] and plotted as a piecewise linear curve.
       - The figure is saved as png file using ResultsDir
       - The DataFrame index is assumed to be a DatetimeIndex if start_date/end_date are None.
    """
    # Compute histogram and CDF
    values = df[field].dropna()
    bins = np.linspace(values.min(), values.max(), 30)
    hist, bin_edges = np.histogram(values, bins=bins, density=False)
    cdf = np.cumsum(hist) / len(values)  # Normalized CDF

    # Display
    fig, (ax1, ax2) = plt.subplots(nrows=2, ncols=1,
                                   sharex=True, figsize=(10, 8),
                                   gridspec_kw={"height_ratios": [2,2]}  # Histogram takes 2/3, CDF takes 1/3)
                                   )

    # 1. Histogram
    # plt.hist(df["GoldPrice"], bins=30, color="salmon", edgecolor="white")
    ax1.bar(bin_edges[:-1], hist,
        width=np.diff(bin_edges), align="edge",
        color="salmon", alpha=0.6,
        edgecolor="black", label=f"Histogramme ({field})",
    )
    ax1.set_title(f"Histogramme (PDF) de {field}, {get_period_as_string(df, start_date, end_date)} ({unit})")
    ax1.set_ylabel("Effectif", fontsize=10)
    ax1.grid(axis='y', alpha=0.2)
    ax1.legend(fontsize=10)
    # Show stats on figure
    text_stats = (
        f"$N$ = {len(df)}\n"
        f"Min = {np.min(df):.2f}\n"
        f"$P_{25}$ = {np.percentile(df, 25):.2f}\n"
        f"Médiane = {np.median(df):.2f}\n"
        f"$P_{75}$ = {np.percentile(df, 75):.2f}\n"
        f"Max = {np.max(df):.2f}\n"
        f"Moyenne = {np.mean(df):.2f}\n"
        f"Écart-type = {np.std(df, ddof=1):.2f}"
    )
    ax1.text(0.85, 0.8,
        text_stats,
        transform=ax1.transAxes,
        fontsize=9,
        bbox=dict(facecolor='white', edgecolor='black', alpha=0.8),
        verticalalignment='top',
        horizontalalignment='center',
    )

    # CDF
    ax2.plot(bin_edges, np.insert(cdf, 0, 0), color="red",
        linewidth=2, label=f"Fonction de répartition (CDF) - {field}"
    )
    ax2.set_title(f"CDF (fonction de répartition) de {field}, {get_period_as_string(df, start_date, end_date)} ({unit})")
    ax2.set_ylabel("Cumulative Probability", fontsize=10)
    ax2.set_xlabel(f"{unit}")
    ax2.grid(True, alpha=0.3)
    ax2.legend(fontsize=12)
    plt.tight_layout()

    results.save_fig(f"ex1_Histogramme_{field}.png", bbox_inches="tight")
    plt.show()

plot_histogram(gold_price, "GoldPrice", "$/once")


## <a id='def_période'> </a>
Comme la période d'observation est longue, nous allons travailler sur une pédiode limitée pour avoir un calcul quasi-instantanné. Mettons-nous dans la peau d'un spéculateur des années 1980, et considérons les données entre Juillet 1982 et Février 1983 pour calculer notre premier variogramme.

In [ ]:
# Slice data [You can change the start and end dates]
start_date = "1982-07-01"
end_date = "1983-02-15"

elapsed = pd.to_datetime(end_date) - pd.to_datetime(start_date)
print(f"La durée de la période considérée est de {elapsed.days} jours")

gold_price_sliced = gold_price.loc[start_date:end_date]

## 2. Calcul du variogramme expérimental (ou empirique)

Nous pouvons maintenant choisir les paramètres de calcul du variogramme empirique:
 - le nombre d'intervalles (`nlags`)
 - la taille de chaque intervalle (`dlag`)
 - la tolérance (`toldis`), en pourcentage de `dlag' (prendre 0.5 pour une couverture complète des distances)

Une fois ces paramètres choisis, on peut exécuter la cellule pour lancer le calcul et visualiser le variogramme.

In [ ]:
# Create the gstlearn 1D database
gl.defineDefaultSpace(gl.ESpaceType.RN, 1)
gl_data = gl.Db_fromPandas(gold_price_sliced)
gl_data.setLocator("Days", gl.ELoc.X)  # 1D coordinate
gl_data.setLocator("GoldPrice", gl.ELoc.Z)  # Variable of interest

# Variogram computation parameters [You need to define them!]
number_of_lags =
lag_size =

params_vario_prix_or = gl.VarioParam.createOmniDirection(nlag=number_of_lags, dlag=lag_size, toldis=0.5)

empirical_variogram_gold = gl.Vario.computeFromDb(
    varioparam=params_vario_prix_or,
    db=gl_data
)

gsutils.plot_1D_variogram(empirical_variogram_gold, title=f"Variogramme expérimental de l'or {get_period_as_string(gold_price_sliced)}",
                          filename=f"ex1_variogramme_expérimental_GoldPrice.png",)


1. Concernant le nombre de paires, qu'observez-vous ? Avez-vous une explication ?
2. Décrivez le comportement du variogramme. Est-il attendu ou surprenant ?

Pour comprendre, nous allons également afficher la série de donnée sur la période retenue.

In [ ]:
gold_price_sliced["GoldPrice"].plot(color='salmon', linewidth=1, figsize=(12, 4),
                                    title=f"GoldPrice {get_period_as_string(gold_price_sliced)} ($/once)")
plt.xlabel("Date")

# Alternative visualization
# plt.figure(figsize=(12, 4))
# plt.plot(gold_price_sliced["Date"], gold_price_sliced["GoldPrice"], color='salmon', linewidth=1)
# plt.title("Prix de l'or de 1982 à 1997")
# plt.xlabel("Date")
# plt.ylabel("Prix ($/once)")
# plt.grid(alpha=0.3)
# plt.tight_layout()
# plt.show()
results.save_fig("ex1_serie_temporelle_GoldPrice.png", dpi=150, bbox_inches="tight")


**Au vu des données, pourquoi le variogramme dépasse-t-il la portée ?**

Pour nous placer dans un cadre plus favorable, vous allons à présent afficher toute la série et choisir une période d'observation plus adaptée.

In [ ]:
gold_price["GoldPrice"].plot(color='salmon', linewidth=1, figsize=(12, 4),
                             title=f"GoldPrice {get_period_as_string(gold_price_sliced)} ($/once)")
plt.xlabel("Date")
results.save_fig("ex1_serie_temporelle_GoldPrice_complète.png", dpi=150, bbox_inches="tight")


**Quelle période vous semble adaptée ?** Vous pouvez la changer [ici](#choose_period) et reprendre les calculs pour vérifier.

## 3. Modélisation du variogramme
### Calage manuel

**L'objectif est maintenant de trouver les paramètres d'un modèle de variogramme honorant approximativement le variogramme empirique.**

In [ ]:
interactive_fit = gsutils.InteractiveVariogramFit(empirical_variogram_gold, "$/once")
interactive_fit.interact()
fitted_vario_model = interactive_fit.model


In [ ]:
print(fitted_vario_model)

### Calage automatique
Pour faciliter la tâche des utilisateurs, gstlearn permet un calage automatique du modèle de variogramme, qui minimise l'écart entre le modèle et le variogramme empirique.
On peut choisir de caler plusieurs types de fonctions mathématiques, par exemple ci-dessous en combinant un modèle Gaussien et un modèle sphérique.

In [ ]:
autofit_model = gl.Model()
autofit_model.fit(empirical_variogram_gold, types=[gl.ECov.SPHERICAL, gl.ECov.GAUSSIAN])
gsutils.plot_1D_variogram(empirical_variogram_gold, title=f"Variogramme expérimental de l'or {get_period_as_string(gold_price_sliced)}",
                              filename=f"ex1_variogramme_automatique_GoldPrice.png", model=autofit_model)
print(autofit_model)

Vous pouvez ajouter des contraintes pour le calage de modèles, voir par exemple [la documentation de gstlearn](https://soft.mines-paristech.fr/gstlearn/courses-latest/python/04_Variography.html)

**Y-a-t-il des différences entre votre modèle calé interactivement et le modèle calé automatiquement ?**